# E2-T05/T06 — full CAB and BBQ runs

Run on **NVIDIA L4**, after committing and pushing this notebook and updating the Drive copy of `LLM_Bias_Audit_PRD.md`. The first setup cells renew the versioned v3 data gate. M1 and M2 can run in separate Colab sessions: remount Drive and repeat setup in each fresh session. Each CAB run checkpoints every 100 prompts; each BBQ run checkpoints every ten items. If disconnected, rerun the interrupted cell with the same Git commit. Do not inspect aggregate fairness results or change the frozen protocol midway.
Before each model’s production cells, run its 20-prompt determinism preflight.


## 1. Mount Drive, pull the locked code, and install the GPU environment


In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys

drive.mount("/content/drive")
SHARED = Path("/content/drive/MyDrive/CMPE253-Project").resolve()
assert SHARED.is_dir(), f"Missing project folder: {SHARED}"
REPO = Path("/content/cmpe253-llm-demographic-bias-audit")
URL = "https://github.com/SJSUF25/cmpe253-llm-demographic-bias-audit.git"
if (REPO / ".git").is_dir():
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)
else:
    subprocess.run(["git", "clone", URL, str(REPO)], check=True)
assert not subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO, text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["make", "setup-colab"], cwd=REPO, check=True)
GPU = subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip()
assert GPU == "NVIDIA L4", f"Primary study requires NVIDIA L4; found {GPU!r}"
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
print("GPU:", GPU, "commit:", COMMIT, "Drive:", SHARED)


## 2. Renew the data gate and check the pilot evidence

If this cell reports a Drive PRD mismatch, replace the Drive copy with the updated local `LLM_Bias_Audit_PRD.md` and rerun it. The command leaves the earlier v1/v2 receipts untouched.


In [ ]:
subprocess.run(["make", "ai-review-gate", f"ROOT={SHARED}"], cwd=REPO, check=True)
GATE = SHARED / "manifests/data_gate_approval_v3.json"
assert GATE.is_file()
PILOT = SHARED / "manifests/pilot_v2/pilot_report.json"
pilot = json.loads(PILOT.read_text())
assert pilot["checks"]["all_eight_cells_complete"] and pilot["checks"]["zero_errors"]
assert (REPO / "docs/pilot_v2_review.md").is_file()
print("v3 data gate:", GATE, "pilot cells:", pilot["run_count"])


## M1 full-run determinism preflight

Run this in every new runtime before this model’s full cells. It repeats the frozen 20-prompt fixture in reverse execution order and checks repeated BBQ likelihood scores. A failure stops this model’s production run.


In [ ]:
PREFLIGHT_M1 = SHARED / f"manifests/full_preflight/m1-{COMMIT[:12]}.json"
subprocess.run(["make", "smoke-model", "MODEL=M1", f"OUTPUT={PREFLIGHT_M1}"], cwd=REPO, check=True)
preflight_m1 = json.loads(PREFLIGHT_M1.read_text())
assert preflight_m1["status"] == "pass"
assert preflight_m1["fixture"]["prompt_count"] == 20
assert preflight_m1["fixture"]["batch_order_reversed_on_second_pass"]
assert preflight_m1["fixture"]["exact_output_match"]
assert preflight_m1["fixture"]["likelihood_max_absolute_delta"] <= 1e-4
assert preflight_m1["fixture"]["likelihood_selected_answers_match"]
assert preflight_m1["environment"]["gpu"]["name"] == "NVIDIA L4"
print("M1 preflight passed:", PREFLIGHT_M1)


## 3. Run M1 CAB explicit and implicit cells

This cell can take hours. Rerun it after a disconnect; verified checkpoints are reused. All four conditions and modes must finish before the model is complete.


In [ ]:
for run_id in ("CAB-EXP-M1-B0", "CAB-EXP-M1-F1", "CAB-IMP-M1-B0", "CAB-IMP-M1-F1"):
    print("Starting", run_id, flush=True)
    subprocess.run(["make", "run-cab", f"ROOT={SHARED}", f"RUN_ID={run_id}", "SPLIT=test"], cwd=REPO, check=True)
    manifest = json.loads((SHARED / "manifests/inference" / f"{run_id}.json").read_text())
    assert manifest["status"] == "complete" and manifest["completed_rows"] == manifest["expected_rows"] == manifest["successful_rows"]
    assert manifest["error_summary"] == {} and manifest["environment"]["gpu"]["name"] == "NVIDIA L4"
    print("Completed", run_id, "rows", manifest["completed_rows"], flush=True)


## 4. Run M1 BBQ baseline and mitigation

Each run scores 1,500 items across three option orders. Rerun after a disconnect to reuse complete ten-item partitions.


In [ ]:
for run_id in ("BBQ-M1-B0", "BBQ-M1-F1"):
    print("Starting", run_id, flush=True)
    subprocess.run(["make", "run-bbq", f"ROOT={SHARED}", f"RUN_ID={run_id}"], cwd=REPO, check=True)
    receipt = json.loads((SHARED / "manifests/bbq" / f"{run_id}-receipt.json").read_text())
    assert receipt["status"] == "complete" and receipt["gpu_name"] == "NVIDIA L4"
    print("Completed", run_id, flush=True)


## M2 full-run determinism preflight

Run this in every new runtime before this model’s full cells. It repeats the frozen 20-prompt fixture in reverse execution order and checks repeated BBQ likelihood scores. A failure stops this model’s production run.


In [ ]:
PREFLIGHT_M2 = SHARED / f"manifests/full_preflight/m2-{COMMIT[:12]}.json"
subprocess.run(["make", "smoke-model", "MODEL=M2", f"OUTPUT={PREFLIGHT_M2}"], cwd=REPO, check=True)
preflight_m2 = json.loads(PREFLIGHT_M2.read_text())
assert preflight_m2["status"] == "pass"
assert preflight_m2["fixture"]["prompt_count"] == 20
assert preflight_m2["fixture"]["batch_order_reversed_on_second_pass"]
assert preflight_m2["fixture"]["exact_output_match"]
assert preflight_m2["fixture"]["likelihood_max_absolute_delta"] <= 1e-4
assert preflight_m2["fixture"]["likelihood_selected_answers_match"]
assert preflight_m2["environment"]["gpu"]["name"] == "NVIDIA L4"
print("M2 preflight passed:", PREFLIGHT_M2)


## 5. Run M2 CAB explicit and implicit cells

This cell can take hours. Rerun it after a disconnect; verified checkpoints are reused. All four conditions and modes must finish before the model is complete.


In [ ]:
for run_id in ("CAB-EXP-M2-B0", "CAB-EXP-M2-F1", "CAB-IMP-M2-B0", "CAB-IMP-M2-F1"):
    print("Starting", run_id, flush=True)
    subprocess.run(["make", "run-cab", f"ROOT={SHARED}", f"RUN_ID={run_id}", "SPLIT=test"], cwd=REPO, check=True)
    manifest = json.loads((SHARED / "manifests/inference" / f"{run_id}.json").read_text())
    assert manifest["status"] == "complete" and manifest["completed_rows"] == manifest["expected_rows"] == manifest["successful_rows"]
    assert manifest["error_summary"] == {} and manifest["environment"]["gpu"]["name"] == "NVIDIA L4"
    print("Completed", run_id, "rows", manifest["completed_rows"], flush=True)


## 6. Run M2 BBQ baseline and mitigation

Each run scores 1,500 items across three option orders. Rerun after a disconnect to reuse complete ten-item partitions.


In [ ]:
for run_id in ("BBQ-M2-B0", "BBQ-M2-F1"):
    print("Starting", run_id, flush=True)
    subprocess.run(["make", "run-bbq", f"ROOT={SHARED}", f"RUN_ID={run_id}"], cwd=REPO, check=True)
    receipt = json.loads((SHARED / "manifests/bbq" / f"{run_id}-receipt.json").read_text())
    assert receipt["status"] == "complete" and receipt["gpu_name"] == "NVIDIA L4"
    print("Completed", run_id, flush=True)


## 7. Reconcile all full cells

Run after both models finish. Exact row counts and manifest hashes must pass. This reports counts and runtime; statistical fairness results are computed later under the registered analysis plan.


In [ ]:
subprocess.run(["make", "report-bbq", f"ROOT={SHARED}"], cwd=REPO, check=True)
bbq_report = json.loads((SHARED / "manifests/bbq/full_report.json").read_text())
assert bbq_report["status"] == "complete" and len(bbq_report["runs"]) == 4
print("BBQ:", [(row["run_id"], row["items"], row["order_disagreement_count"]) for row in bbq_report["runs"]])
import hashlib
commits = set()
environments = {"M1": set(), "M2": set()}
for model in ("M1", "M2"):
    for mode in ("EXP", "IMP"):
        for condition in ("B0", "F1"):
            run_id = f"CAB-{mode}-{model}-{condition}"
            manifest = json.loads((SHARED / "manifests/inference" / f"{run_id}.json").read_text())
            output = SHARED / manifest["output_path"]
            assert manifest["status"] == "complete" and manifest["completed_rows"] == manifest["expected_rows"] == manifest["successful_rows"]
            assert manifest["error_summary"] == {} and hashlib.sha256(output.read_bytes()).hexdigest() == manifest["output_sha256"]
            assert manifest["environment"]["gpu"]["name"] == "NVIDIA L4"
            commits.add(manifest["code_commit"])
            env = manifest["environment"]
            environments[model].add((env["gpu"]["name"], env["gpu"]["cuda_runtime"], env["packages"]["torch"], env["packages"]["transformers"], manifest["package_lock_sha256"]))
            print(run_id, "rows", manifest["completed_rows"], "seconds", round(manifest["elapsed_seconds"]))
assert commits == {bbq_report["code_commit"]} == {COMMIT}
assert all(len(values) == 1 for values in environments.values())
print("All 12 full CAB/BBQ cells reconciled at", COMMIT)
